In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report,accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd


df = pd.read_csv('insurance.csv')
df.sample(5)

df_feature = df.copy()

# Feature 1: BMI calculation

df_feature["bmi"] = df_feature["weight"] / (df_feature["height"] ** 2)

# Feature 2: Age group calculation

def age_group(age):
    if age < 25:
        return "Young"
    elif age < 45:
        return "Adult"
    elif age < 60:
        return "middle_aged"
    else:
        return "senior"


df_feature["age_group"] = df_feature["age"].apply(age_group)


# Feature 3: Lifestyle Risk

def lifestyle_risk(row):
    if row["smoker"] and row["bmi"] > 30:
        return "high"
    elif row["smoker"] and row["bmi"] > 27:
        return "medium"
    else:
        return "low"

df_feature["lifestyle_risk"] = df_feature.apply(lifestyle_risk, axis=1)

tier_1_cities = ["Delhi","Mumbai","Kolkata","Chennai","Bangalore","Hyderabad","Pune", "Ahmedabad"]
tier_2_cities = ["Jaipur",
    "Lucknow",
    "Kanpur",
    "Nagpur",
    "Indore",
    "Bhopal",
    "Coimbatore",
    "Kochi",
    "Surat",
    "Vadodara",
    "Visakhapatnam",
    "Patna",
    "Bhubaneswar",
    "Chandigarh",
    "Madurai",
    "Mysuru",
    "Rajkot",
    "Guwahati",
    "Trichy",
    "Vijayawada"]

# Feature 4 city Tier

def city_tier(city):
    if city in tier_1_cities:
        return 1
    elif city in tier_2_cities:
        return 2
    else:
        return 3

df_feature["city_tier"] = df_feature["city"].apply(city_tier)

df_feature.drop(columns=['age','weight','height','smoker','city'])[['income_lpa','occupation','bmi','age_group','lifestyle_risk','city_tier','insurance_premium_category']]

X = df_feature[['bmi','age_group','lifestyle_risk','city_tier','income_lpa','occupation']]
y = df_feature["insurance_premium_category"]


# define categorical and numerical features

categorical_features = ['age_group','lifestyle_risk','income_lpa','occupation','city_tier']
numeric_features = ['bmi','income_lpa']

categorical_features = [2]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="passthrough"
)


#Create column transformer for One Hot encoder

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(),categorical_features),
        ("num","passthrough",numeric_features)
    ]
)

#Create a pipeline with preprocessing and random forest classifier

data_pipeline = Pipeline(steps=[
    ("preprocessor",preprocessor),
    ("classifier",RandomForestClassifier(random_state=42))
])

# Split data and train model

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=1)
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), ['age_group','lifestyle_risk','occupation']),
        ("num", "passthrough", ['bmi','city_tier','income_lpa'])
    ]
)

data_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

data_pipeline.fit(X_train, y_train)


# Predict and evaluate

y_pred = data_pipeline.predict(X_test)
accuracy_score(y_test,y_pred)


import pickle
#Save the trained pipelines using pickle

pickle_model_path = "model.pkl"
with open(pickle_model_path,"wb") as f:
    pickle.dump(data_pipeline,f)
